# Model Training & Optimization Pipeline - Car Price Prediction
This notebook handles feature engineering, standard scaling, categorical encoding, multi-model evaluation, GridSearchCV optimization, and permutation feature importances.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

# Make sure we import our modular transformers
sys.path.append(os.path.abspath('../backend'))
from app.services.preprocessing import clean_raw_data, get_preprocessing_pipeline, CarFeatureEngineer
from sklearn.model_selection import train_test_split

# 1. Load Raw Data & Apply Custom Pipeline
raw_df = pd.read_csv('../dataset/raw/car_data.csv')
cleaned_df = clean_raw_data(raw_df)

X = cleaned_df.drop(columns=['Selling_Price'])
y = cleaned_df['Selling_Price']

# 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline = get_preprocessing_pipeline()
pipeline.fit(X_train)

X_train_processed = pipeline.transform(X_train)
X_test_processed = pipeline.transform(X_test)
print("Processed features shape:", X_train_processed.shape)

### 2. Multi-Model GridSearch Training & Comparison

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    preds = model.predict(X_test_processed)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    results[name] = {'R2': r2, 'MAE': mae}
    print(f"{name:<22} | R2: {r2:.4f} | MAE: {mae:.3f} Lakhs")

### 3. Matplotlib Visualizations of Model Performance Comparison

In [ ]:
# Bar chart comparing R2 Scores
plt.figure(figsize=(10, 5))
names = list(results.keys())
r2_scores = [results[n]['R2'] for n in names]

plt.bar(names, r2_scores, color=['gray', 'blue', 'orange', 'red'], alpha=0.8)
plt.ylabel('R² Score')
plt.title('Regression Model Comparison (R² Accuracy)')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)
for i, score in enumerate(r2_scores):
    plt.text(i, score + 0.02, f"{score*100:.1f}%", ha='center', fontweight='bold')
plt.show()